<a href="https://colab.research.google.com/github/codeBMB/Oct26_codeBMB/blob/main/004_Biopython_sequence_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img align="right" src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/PyBMB_logo.png" width="200"/>

#Project: CodeBMB: Computational Literacy for Biochemistry and Molecular Biology Education
## Notebook: Biopython and Sequence Analysis

This notebook introduces core BioPython modules through a repeating four-part cycle:

| Step | What you do |
|---|---|
| **Show** | Run a minimal working example with real biological data |
| **Inspect** | Ask: what type is this? What attributes does it have? |
| **Connect** | Connect the new tool to pandas, strings, or loops you already know |
| **Challenge** | Apply the concept to a new sequence or dataset |

**By the end of this notebook you will be able to:**
- *Work with biological sequences in Python:* create Seq and SeqRecord objects, apply biological methods (transcribe, translate, GC content), and explain how a Seq object differs from a plain Python string.
- *Retrieve sequences from NCBI:* use Entrez.efetch() and Entrez.esearch() to fetch and search sequence records directly from Python without using the NCBI website.
- *Read biological file formats and parse BLAST results:* load a FASTA file with SeqIO.parse(), navigate the alignment/HSP structure of BLAST output, and extract key metrics (score, e-value, percent identity).
- *Connect sequence data to pandas and matplotlib:* convert lists of SeqRecord objects or BLAST hits into a DataFrame and immediately apply filtering, summary statistics, and visualization.
- *Define a reusable function* that takes a biological object as input and returns a structured output.
- *Use the return-type reference table* to identify what any BioPython function produces and know what to do with the result.



**Libraries:**
* `pandas` - convert sequence data to DataFrames
* `matplotlib` - histograms and scatter plots
* `Biopython` - Biological string objects, read/write FASTA files, connect to NCBI databases, perform BLAST searches and parse XML output

**Status:** In development for the October 2026 Workshop

**License:**

<img src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/by-nc-sa.png" width="100"/>

CC BY-NC-SA — free to use and adapt for non-commercial educational purposes with attribution.

---
**Authors:** Wally Novak

**Acknowledgements:** Supported by NSF IUSE 2518732 and 2518733.

**Contact:** codingforBMB [at] gmail [dot] com


---

# Section 0a. How to run this notebook.

To run the cell below, simply click the Run button located in that cell.

![run button image](https://github.com/wallynovak/biochemistry_seq_analysis/blob/main/images/run.png?raw=1)

NOTE: A cell is still running if you see a "stop" button with a moving circle around it. You can tell a cell has completed running as it will have a number in brackets on the left hand side (e.g. [1]) and a checkmark with the amount of time it took to run underneath it.

Please ensure every cell is done running before running the next cell.

1. **Work through each section in order** as each section builds on the previous one.
2. **Challenge** cells are marked with that heading. You can write your code there.
3. The **Reference Table** at the end summarizes what the major functions return.

> **When you get an error:** Read the last line of the error message first.
> It usually identifies the problem.

---
## Section 0b. Environment Setup and Libraries

Run the two cells below before starting.
They install BioPython and import everything this notebook uses.
You only need to install once per Colab session.

In [ ]:
!pip install biopython --quiet

In [ ]:
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction   # BioPython >= 1.80; older versions: from Bio.SeqUtils import GC
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO, Entrez
from Bio.Blast import NCBIWWW, NCBIXML

import pandas as pd
import matplotlib.pyplot as plt
import urllib.request

print("All imports loaded successfully.")

---
## Section 1 -- `Seq`: The Biological String

### What problem does this solve?

A plain Python string knows nothing about biology.
A `Seq` object knows it is a DNA, RNA, or protein sequence
and has biological methods built in -- while still behaving
exactly like a string for slicing, indexing, and iteration.

*Example question:* We have the coding sequence for a fragment of gene X.
What can Python tell us about it without querying any database?

### Show

In [ ]:
dna = Seq("ATGAAACCCGGGTTTTAA")
print(dna)
print(type(dna))

### Inspect -- what can a `Seq` object do?

In [ ]:
# Length and slicing work exactly like a Python string
print(len(dna))          # 18
print(dna[0:3])          # ATG -- start codon
print(dna[-3:])          # TAA -- stop codon

# GC content
print(f"GC content: {gc_fraction(dna):.1%}")

# Biological methods
print(dna.reverse_complement())
print(dna.transcribe())      # DNA -> mRNA  (T becomes U)
print(dna.translate())       # DNA -> protein

In [ ]:
# translate() returns another Seq object
protein = dna.translate()
print(type(protein))         # <class 'Bio.Seq.Seq'>
print(protein)               # MKPG*
print(protein[:-1])          # slice off the stop codon

### Connect -- string operations you already know work the same

In [ ]:
# Membership test
print("ATG" in dna)

# Count occurrences
print(dna.count("G"))

# Iterate character by character
for base in dna[:6]:
    print(base)

### Challenge 1

Given this coding sequence:
`ATGGCCATGGCGCCCAGAACTGAGATCAATAGTACCCGTATTAACGGGTGA`

1. Store it as a `Seq` object called `challenge_seq`.
2. Print the length.
3. Print the start codon (first 3 bases) and stop codon (last 3 bases) using slicing.
4. Translate the sequence and store as `challenge_protein`. Print it.
5. Count how many times the codon `ATG` appears in `challenge_seq`.

> **Hint:** `Seq.count("ATG")` works the same as `str.count("ATG")`.

In [ ]:
# Challenge 1 -- your code here


---
## Section 2 -- `SeqRecord`: Sequence Plus Metadata

### What problem does this solve?

A `Seq` object is just the letters.
In real biology, a sequence has an accession number, a species name,
and a description. A `SeqRecord` wraps the sequence and its metadata
into one object with named attributes.

### Show

In [ ]:
record = SeqRecord(
    Seq("MKPG"),
    id          = "demo_001",
    description = "Demo fumarase fragment, E. coli"
)
print(record)

### Inspect

In [ ]:
print(type(record))           # <class 'Bio.SeqRecord.SeqRecord'>

# The three attributes you will use constantly
print(record.id)              # demo_001
print(record.description)     # Demo fumarase fragment, E. coli
print(record.seq)             # MKPG -- this is a Seq object
print(type(record.seq))       # <class 'Bio.Seq.Seq'>
print(len(record))            # 4

### Connect -- `SeqRecord` as a row in a pandas DataFrame

Think of a `SeqRecord` as a row in a spreadsheet.
The bridge to pandas is a simple dictionary conversion.

In [ ]:
# Convert one SeqRecord to a dictionary -- one future row in a DataFrame
row = {
    "id":          record.id,
    "description": record.description,
    "length":      len(record.seq),
    "sequence":    str(record.seq)    # convert Seq -> plain string for pandas
}
print(row)

# Build a one-row DataFrame to preview the structure
df_demo = pd.DataFrame([row])
df_demo

### Challenge 2

1. Create a `SeqRecord` for human insulin using the first 10 amino acids: `MALWMRLLPL`.
   Use `"INS_HUMAN"` as the ID and write a brief, realistic description.
2. Print each attribute (`id`, `description`, `seq`, length) on its own line
   using an f-string.
3. Build a dictionary from the record and print it.

In [ ]:
# Challenge 2 -- your code here


---
## Section 3 -- `SeqIO`: Reading and Writing Sequence Files

### What problem does this solve?

`SeqIO` is the standard interface for reading and writing biological file formats.
One function -- `SeqIO.parse()` -- handles FASTA, GenBank, FASTQ, and dozens of
other formats.

### Show

In [ ]:
# Download pre-computed BLAST results from GitHub
fasta_url = "https://raw.githubusercontent.com/codeBMB/October26/main/data/fumc_seqs.fasta"
urllib.request.urlretrieve(fasta_url, "fumc_seqs.fasta")

# Parse the FASTA file into a list of SeqRecord objects
records = list(SeqIO.parse("fumc_seqs.fasta", "fasta"))
print(f"Number of sequences: {len(records)}")

### Inspect -- generators and `list()`

This is the most important concept in `SeqIO`:
`SeqIO.parse()` returns a **generator**, not a list.
A generator reads one record at a time rather than loading the entire file into memory.
Wrapping it in `list()` forces all records to be read immediately.

In [ ]:
# What does SeqIO.parse() return BEFORE calling list()?
raw_parser = SeqIO.parse("fumc_seqs.fasta", "fasta")
print(type(raw_parser))         # a generator object -- lazy, reads one at a time

# list() forces all records to be generated immediately
records = list(raw_parser)
print(type(records))            # <class 'list'>
print(type(records[0]))         # <class 'Bio.SeqRecord.SeqRecord'>

# Access individual records by index
print(records[0].id)
print(records[0].description)
print(f"Length: {len(records[0].seq)} aa")

### Connect -- defining a function, then building a DataFrame

In the line fitting notebook you wrote something like:

    def linear(x, m, b):
        return m * x + b

The same pattern applies here, you pass variables in and one output comes out.
We will define a function first, verify it works on a single record,
then call it across the full list to build a DataFrame.

In [ ]:
# Define the function
def record_to_dict(record):
    """Convert a SeqRecord to a dictionary suitable for a DataFrame row."""
    return {
        "id":          record.id,
        "description": record.description,
        "length":      len(record.seq),
        "sequence":    str(record.seq)
    }

# Test it on one record before using it on all of them
print(record_to_dict(records[0]))

In [ ]:
# Call the function across all records to build the DataFrame
data = []
for record in records:
    data.append(record_to_dict(record))

df = pd.DataFrame(data)
print(df.shape)
df.head()

In [ ]:
# All Day 1 pandas skills now apply to sequence data
print(df["length"].describe())
print(f"\nSequences between 450 and 470 aa: {df['length'].between(450, 470).sum()}")

### Challenge 3

1. Modify `record_to_dict()` to add a fifth key, `gc_content`, that stores
   the GC fraction of each sequence rounded to three decimal places.
   Rebuild the DataFrame and confirm the new column appears in `df.head()`.
2. Plot a histogram of sequence lengths using `df["length"]`.
   Add axis labels and a title that includes the total number of sequences.
3. Filter `df` to sequences between 450 and 470 aa. Store as `df_filtered`.
   Print how many sequences remain.
4. (Stretch goal) Convert filtered rows back to SeqRecord objects and write
   them to `fumc_filtered.fasta` using `SeqIO.write()`.

Hint for item 1: `gc_fraction()` takes a Seq object, not a string.
Use `record.seq` not `record_to_dict(record)["sequence"]`.


In [ ]:
# Challenge 3 -- your code here


---
## Section 4 -- `Bio.Entrez`: Connecting to NCBI

### What problem does this solve?

`Entrez` automates what you would otherwise do manually on the NCBI website, such as searching for sequences and downloading records directly from Python
without clicking through menus or copying sequences by hand.

### Show

In [ ]:
Entrez.email = "your_email@institution.edu"    # required by NCBI

handle = Entrez.efetch(
    db      = "sequences",
    id      = "82407514",       # GI number for bacterial fumarase, PDB: 1YFE
    rettype = "fasta",
    retmode = "text"
)
bacterial_fumarase = SeqIO.read(handle, "fasta")
handle.close()

print(bacterial_fumarase.id)
print(bacterial_fumarase.description)
print(f"Length: {len(bacterial_fumarase.seq)} aa")

### Inspect -- what is a handle?

`Entrez.efetch()` returns a **handle** -- a connection to NCBI's server,
not the data itself. It behaves exactly like Python's built-in `open()` for local files.

The pattern is always: **open -> read -> close**

In [ ]:
# Open a fresh handle and inspect the raw content
handle = Entrez.efetch(
    db="sequences", id="82407514", rettype="fasta", retmode="text"
)
print(type(handle))         # a file-like object -- same concept as open()

# Read as raw text to see the underlying format
raw_text = handle.read()
print(raw_text[:300])       # first 300 characters in FASTA format
handle.close()
# Once read, a handle is empty -- always close after reading

### Connect -- `Entrez.esearch()` finds accession numbers programmatically

In [ ]:
search_handle = Entrez.esearch(
    db   = "protein",
    term = "fumarase[Title] AND Escherichia coli[Organism]"
)
search_results = Entrez.read(search_handle)
search_handle.close()

print(f"Number of hits: {search_results['Count']}")
print(f"First 5 IDs:    {search_results['IdList'][:5]}")

### Challenge 4

1. Fetch the human fumarase sequence using GI number `196049942`.
   Store it as `human_fumarase`.
2. Print the sequence length, the first 30 amino acids, and the last 30 amino acids.
3. Write both `bacterial_fumarase` and `human_fumarase` to a single FASTA file
   called `two_fumarases.fasta` using `SeqIO.write()`.

> **Hint:** Pass a list of `SeqRecord` objects as the first argument to `SeqIO.write()`.

In [ ]:
# Challenge 4 -- your code here


---
## Section 5 -- `Bio.Blast`: Running and Parsing BLAST Searches

### What problem does this solve?

`Bio.Blast` lets you run BLAST searches from Python and parse the results
into a structured format -- without using the NCBI website
and without downloading XML files by hand.

### The BLAST query -- shown but not run live

A search with `hitlist_size=10000` takes 20-40 minutes.
We ran this before the workshop and saved the XML output to GitHub.
Read the commented cell below to understand the parameters, then continue.

In [ ]:
# ---------------------------------------------------------------
# BLAST QUERY -- shown for reference, not executed live
#
# query = NCBIWWW.qblast(
#     program      = "blastp",
#     database     = "refseq_protein",
#     sequence     = bacterial_fumarase.seq,
#     hitlist_size = 10000,
#     alignments   = 0,
#     format_type  = "XML"
# )
# with open("fumc_blastp.xml", "w") as f:
#     f.write(query.read())
# query.close()
# ---------------------------------------------------------------
print("BLAST parameters reviewed. Loading pre-computed results below.")

### Show -- parse pre-computed results

In [ ]:
# Download the pre-computed BLAST XML from GitHub
blast_url = "https://raw.githubusercontent.com/codeBMB/October26/main/data/fumc_blastp_clean.xml"
urllib.request.urlretrieve(blast_url, "fumc_blastp_clean.xml")

with open("fumc_blastp_clean.xml") as result_handle:
    blast_records = NCBIXML.parse(result_handle)
    blast_record  = next(blast_records)    # one query sequence = one BLAST record

print(f"Query:          {blast_record.query[:60]}")
print(f"Number of hits: {len(blast_record.alignments)}")

### Inspect -- unpacking the nested structure

BLAST results are nested three levels deep:

    blast_record
      -> alignments        (list of database hits)
           -> alignment
                -> hsps    (High-Scoring Pairs: the aligned regions within a hit)
                     -> hsp

An **alignment** is one database sequence that matched your query.
An **HSP** is the specific aligned region within that match.

In [ ]:
alignment = blast_record.alignments[0]    # first database hit
hsp       = alignment.hsps[0]             # first high-scoring pair within that hit

print(f"Hit title:   {alignment.title[:60]}")
print(f"Score:       {hsp.score}")
print(f"E-value:     {hsp.expect:.2e}")
print(f"Identities:  {hsp.identities}/{hsp.align_length}")
print(f"% Identity:  {hsp.identities / hsp.align_length:.1%}")

### Connect -- convert all BLAST hits to a pandas DataFrame

In [ ]:
hits = []
for alignment in blast_record.alignments:
    for hsp in alignment.hsps:
        hits.append({
            "title":    alignment.title.split(">")[0][:60],
            "score":    hsp.score,
            "evalue":   hsp.expect,
            "identity": hsp.identities / hsp.align_length,
            "length":   hsp.align_length,
            "sequence": hsp.sbjct.replace("-", "")
        })

df_blast = pd.DataFrame(hits)
print(df_blast.shape)
df_blast.head()

In [ ]:
# Day 1 pandas and matplotlib skills apply immediately to BLAST data
df_blast["identity"].hist(bins=50, color="steelblue", edgecolor="white")
plt.xlabel("Percent identity")
plt.ylabel("Count")
plt.title("Distribution of BLAST hit identities")
plt.tight_layout()
plt.show()

print(f"Hits with identity > 0.70: {(df_blast['identity'] > 0.70).sum()}")

### Challenge 5

Using `df_blast`:

1. Create a scatter plot of e-value (y-axis, log scale) versus percent identity (x-axis).
   Use `plt.yscale("log")`. Add axis labels and a title.
2. Filter to hits with identity > 0.70 AND e-value < 1e-10.
   Store as `df_high_quality`. Print how many hits remain.
3. Print the mean and standard deviation of alignment length for `df_high_quality`.

In [ ]:
# Challenge 5 -- your code here


---
## Section 6 -- `Bio.PDB` and `Bio.Phylo`: Orientation

These modules are introduced here so you know they exist and where to find them.
Full treatment of each requires a dedicated session.

---

### `Bio.PDB` -- 3D protein structures

Parses PDB and mmCIF files into a four-level data model:
`Structure -> Model -> Chain -> Residue -> Atom`

Each atom stores a name, element, and x/y/z coordinates.
Use `Bio.PDB` to calculate inter-residue distances, identify active site residues,
or count contacts at a protein-protein interface.

**Entry point:**

    from Bio.PDB import PDBParser
    parser    = PDBParser(QUIET=True)
    structure = parser.get_structure("1YFE", "1yfe.pdb")
    for chain in structure[0]:
        print(chain.id, len(list(chain.get_residues())))

**Returns:** A `Structure` object you navigate with nested loops.
**Full reference:** BioPython Tutorial, Chapter 11

---

### `Bio.Phylo` -- phylogenetic trees

Reads Newick, Nexus, and PhyloXML tree files.
Use it to draw, traverse, and annotate phylogenetic trees in Python.

**Entry point:**

    from Bio import Phylo
    tree = Phylo.read("tree.nwk", "newick")
    Phylo.draw(tree)

**Returns:** A `Tree` object with `.root` and `.find_clades()`.
**Full reference:** BioPython Tutorial, Chapter 13

Both chapters: https://biopython.org/DIST/docs/tutorial/Tutorial.html

---
## Reference Table -- What Does Each Function Return?

Use this as a quick lookup when you know what function you ran
but are not sure what to do with the result.

| Function | Returns | How to use the result |
|---|---|---|
| `Seq("ATGC")` | `Seq` object | Slice, call methods, pass to `SeqRecord` |
| `seq.translate()` | `Seq` object (protein) | Slice, `str()`, or print |
| `seq.transcribe()` | `Seq` object (mRNA) | Same as above |
| `gc_fraction(seq)` | `float` | Multiply by 100 for percent |
| `SeqRecord(seq, id, description)` | `SeqRecord` object | Access `.seq`, `.id`, `.description` |
| `SeqIO.parse(file, format)` | Generator of `SeqRecord` | Wrap in `list()` or use in a `for` loop |
| `SeqIO.read(handle, format)` | Single `SeqRecord` | Use directly |
| `SeqIO.write(records, file, format)` | `int` (count written) | Check count to confirm success |
| `Entrez.efetch(...)` | File handle | Pass to `SeqIO.read()` or `.read()`, then close |
| `Entrez.esearch(...)` | File handle | Pass to `Entrez.read()`, then close |
| `Entrez.read(handle)` | `dict` | Access keys like `["Count"]` and `["IdList"]` |
| `NCBIWWW.qblast(...)` | File handle | Write to file or pass to `NCBIXML.parse()` |
| `NCBIXML.parse(handle)` | Generator of BLAST records | `next()` for one query, loop for multiple queries |
| `blast_record.alignments` | List of `Alignment` objects | Loop, access `.title` and `.hsps` |
| `alignment.hsps[0]` | `HSP` object | Access `.score`, `.expect`, `.identities`, `.sbjct` |

---
## Wrap-up

### Skills you have built in this notebook

1. Create and manipulate `Seq` and `SeqRecord` objects
2. Read biological sequence files with `SeqIO.parse()` and understand generators
3. Fetch sequences from NCBI with `Entrez.efetch()` and search with `Entrez.esearch()`
4. Parse BLAST XML output and convert hits to a pandas DataFrame
5. Apply Day 1 pandas and matplotlib skills directly to sequence data
6. Know where to find `Bio.PDB` and `Bio.Phylo` when you need them

### Where to go next

**Full fumarase workflow** (BLAST, CD-HIT, and ClustalOmega alignment):
see the complete notebook in the codeBMB October26 repository.

**BioPython Tutorial and Cookbook:** https://biopython.org/DIST/docs/tutorial/Tutorial.html

- Chapter 2: Seq objects
- Chapter 4: SeqRecord
- Chapter 5: SeqIO
- Chapter 9: Entrez
- Chapter 11: Bio.PDB
- Chapter 13: Bio.Phylo

**codeBMB GitHub:** https://github.com/codeBMB

---
<img src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/by-nc-sa.png" width="80"/> This notebook is licensed CC BY-NC-SA.
Contact: codingBMB [at] gmail [dot] com